# Orchestrator-Workers: el manager decide sobre la marcha

Clasificación: **Proceso jerárquico.** Un manager LLM lee la petición del cliente y decide qué agentes necesita, en qué orden y cuánto trabajo darle a cada uno.

A diferencia de parallelization (donde las 4 tasks están fijadas antes de arrancar), aquí el manager adapta la ejecución al caso concreto. Un cliente que solo quiere relajarse necesita más trabajo de actividades; uno con itinerario ajustado, más de vuelos.

## Cómo funciona en CrewAI

`Process.hierarchical` activa un manager automático que orquesta a los agentes. El manager recibe la task principal, decide a quién delegar, y puede volver a consultar al mismo agente si necesita más detalle.

```python
crew = Crew(
    agents=[...],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
)
```

In [22]:
!uv pip install -r requirements.txt --quiet

In [23]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

In [28]:
from crewai import Task, Crew, Process
from crewai.events.event_bus import crewai_event_bus
from crewai.events.types.tool_usage_events import ToolUsageStartedEvent
from rich.console import Console
from rich.table import Table
from viajes_crew import ViajesCrew

base_crew = ViajesCrew()


# boilerplate function for better traceability and understanding
def make_delegation_tracker():
    """Returns (listener_fn, delegations_list).

    Register listener_fn with the event bus before kickoff and unregister it after.
    The delegations list is populated from the background thread during execution.
    """
    delegations = []

    def listener(source, event):
        if event.tool_name in ("delegate_work_to_coworker", "ask_question_to_coworker"):
            args = event.tool_args if isinstance(event.tool_args, dict) else {}
            target_agent = args.get("coworker", "?")
            task_or_q = args.get("task", args.get("question", "?"))[:80]
            delegations.append((event.tool_name, target_agent, task_or_q))

    return listener, delegations


# boilerplate function for better traceability and understanding
def print_delegations(delegations):
    """Renders the delegation summary as a rich table."""
    console = Console()
    table = Table(title=f"Agents invoked by manager ({len(delegations)} delegations)", show_lines=True)
    table.add_column("Tool", style="cyan")
    table.add_column("Delegated to (coworker)", style="green")
    table.add_column("Task / Question", style="white")

    for tool_name, agent_role, task in delegations:
        table.add_row(tool_name, agent_role, task)

    console.print(table)


peticion_cliente = (
    "Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. "
    "Lo unico que de verdad importa es ver auroras boreales y banarnos en fuentes termales; "
    "el resto (vuelos, alojamiento, alquiler de coches, rutas, transporte) lo he revisado ya manualmente."
)

main_task = Task(
    description=(
        f"Peticion del cliente: {peticion_cliente}\n\n"
        "Lee la peticion con atencion. El cliente puede haber indicado que ya ha gestionado ciertos aspectos del viaje. "
        "Delega SOLO a los especialistas necesarios para cubrir lo que el cliente AUN necesita. "
        "No invoques especialistas para aspectos que el cliente ya tiene resueltos."
    ),
    expected_output="Respuesta completa a lo que el cliente ha pedido, cubriendo unicamente los aspectos que todavia necesita.",
)

crew = Crew(
    agents=[base_crew.vuelos(), base_crew.alojamiento(), base_crew.actividades(), base_crew.transporte()],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_llm="gpt-5.5",
    verbose=True,
)

# boilerplate code for better traceability and understanding
listener, delegations = make_delegation_tracker()
crewai_event_bus.on(ToolUsageStartedEvent)(listener)

result = await crew.kickoff_async()

# boilerplate code for better traceability and understanding
crewai_event_bus.off(ToolUsageStartedEvent, listener)
print_delegations(delegations)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.14                                                                                       │
│  Latest version:  1.15.16                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6759e25d-9a71-4fa3-a2ad-4b04bd889113                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento, alquiler de     │
│  coches, rutas, transporte) lo he revisado ya manualmente.                                                      │
│                                                                                                                 │
│  Lee la peticion con atencion. El cliente puede haber indicado que ya ha gestionado ciertos aspectos del        │
│  viaje. Delega SOLO a los especialistas necesarios para cubrir lo que el cliente AUN necesita. No invoques      │
│  especialistas para aspectos que el cliente ya tiene resueltos.                                                 │
│  ID: 4b9592ce-dbde-420f-be2c-77585560d5da                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento, alquiler de     │
│  coches, rutas, transporte) lo he revisado ya manualmente.                                                      │
│                                                                                                                 │
│  Lee la peticion con atencion. El cliente puede haber indicado que ya ha gestionado ciertos aspectos del        │
│  viaje. Delega SOLO a los especialistas necesarios para cubrir lo que el cliente AUN necesita. No invoques      │
│  especialistas para aspectos que el cliente ya tiene resueltos.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#62) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': 'Cliente solicita un viaje a Islandia de 5 días para 2 personas con presupuesto total de     │
│  2200 EUR. Indica explícitamente que lo único que de verdad importa es ver auroras boreales y bañars...         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Aquí tienes una propuesta centrada **solo en lo que realmente os importa**: **ver auroras boreales** y         │
│  **bañaros en fuentes termales** durante 5 días en Islandia, sin entrar en vuelos, alojamiento, coche, rutas    │
│  generales ni transporte, ya que lo tenéis revisado.                                                            │
│                                                                                                                 │
│  ## Objetivo del viaje                                                                                          │
│                                                                                                                 │
│  Para 5 días en Islandia, lo más inteligente es combinar:                                                       │
│                                                                                                                 │
│  1. **Varias oportunidades nocturnas de auroras**, porque dependen totalmente del cielo despejado y la          │
│  actividad solar.                                                                                               │
│  2. **Una fuente termal o spa geotermal casi cada día**, alternando experiencias famosas con opciones más       │
│  auténticas y económicas.                                                                                       │
│  3. **Reservas clave con flexibilidad**, especialmente en Blue Lagoon / Sky Lagoon y tours de auroras.          │
│                                                                                                                 │
│  Importante: las auroras se ven mejor entre **septiembre y abril**, especialmente de noche, lejos de luces y    │
│  con cielo despejado. No se pueden garantizar, así que conviene planear **3 o 4 noches potenciales de           │
│  búsqueda**.                                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Plan sugerido de 5 días                                                                                      │
│                                                                                                                 │
│  ## Día 1 — Primer baño termal + primera búsqueda de auroras                                                    │
│                                                                                                                 │
│  ### Baño termal recomendado: Laugarvatn Fontana                                                                │
│                                                                                                                 │
│  **Laugarvatn Fontana** es una muy buena primera experiencia termal: baños calientes, saunas geotermales y      │
│  ambiente relajado junto al lago. Es menos masificado que Blue Lagoon y encaja muy bien si queréis algo cómodo  │
│  pero no excesivamente turístico.                                                                               │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento, alquiler de     │
│  coches, rutas, transporte) lo he revisado ya manualmente.                                                      │
│                                                                                                                 │
│  Lee la peticion con atencion. El cliente puede haber indicado que ya ha gestionado ciertos aspectos del        │
│  viaje. Delega SOLO a los especialistas necesarios para cubrir lo que el cliente AUN necesita. No invoques      │
│  especialistas para aspectos que el cliente ya tiene resueltos.                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                     Agents invoked by manager (1 delegations)                                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Tool                      ┃ Delegated to (coworker)     ┃ Task / Question                                       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ delegate_work_to_coworker │ Especialista en Actividades │ Prepara una propuesta completa, práctica y priorizada │
│                           │                             │ de actividades para 5 días                            │
└───────────────────────────┴─────────────────────────────┴───────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6759e25d-9a71-4fa3-a2ad-4b04bd889113                                                                       │
│  Final Output: Aquí tienes una propuesta centrada **solo en lo que realmente os importa**: **ver auroras        │
│  boreales** y **bañaros en fuentes termales** durante 5 días en Islandia, sin entrar en vuelos, alojamiento,    │
│  coche, rutas generales ni transporte, ya que lo tenéis revisado.                                               │
│                                                                                                                 │
│  ## Objetivo del viaje                                                                                          │
│                                                                                                                 │
│  Para 5 días en Islandia, lo más inteligente es combinar:                                                       │
│                                                                                                                 │
│  1. **Varias oportunidades nocturnas de auroras**, porque dependen totalmente del cielo despejado y la          │
│  actividad solar.                                                                                               │
│  2. **Una fuente termal o spa geotermal casi cada día**, alternando experiencias famosas con opciones más       │
│  auténticas y económicas.                                                                                       │
│  3. **Reservas clave con flexibilidad**, especialmente en Blue Lagoon / Sky Lagoon y tours de auroras.          │
│                                                                                                                 │
│  Importante: las auroras se ven mejor entre **septiembre y abril**, especialmente de noche, lejos de luces y    │
│  con cielo despejado. No se pueden garantizar, así que conviene planear **3 o 4 noches potenciales de           │
│  búsqueda**.                                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Plan sugerido de 5 días                                                                                      │
│                                                                                                                 │
│  ## Día 1 — Primer baño termal + primera búsqueda de auroras                                                    │
│                                                                                                                 │
│  ### Baño termal recomendado: Laugarvatn Fontana                                                                │
│                                                                                                                 │
│  **Laugarvatn Fontana** es una muy buena primera experiencia termal: baños calientes, saunas geotermales y      │
│  ambiente relajado junto al lago. Es menos masificado que Blue Lagoon y encaja muy bien si queréis algo cómodo  │
│  pero no excesivamente turístico.                                                                               │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Qué define este patrón

El manager decide en runtime cuánto delega a cada agente. Puede consultar poco a transporte y volver dos veces a actividades si la petición lo requiere.